# Experiment 01b — hybrid arm at matched budget

The classical grid is done. `matched_classical` (122 params) and `paper_linear`
(26) both die at best_ma50 ~23; `oversized_mlp` (10,934) lives. The result we
want — *does the circuit reach where an equal-budget classical block does not?* —
needs the **hybrid arm run under the exact same protocol** (60k steps, FIX-01
on/off, 3 seeds).

Right now the only hybrid data is the earlier single-seed run at 103k steps that
peaked at 57. Comparing 60k classical against 103k hybrid is not clean. This
notebook closes that gap: it runs `hybrid_fig4` in the same grid, then compares
all four arms at the same 60k budget.

**Cost warning.** These are PQC runs (adjoint differentiation, 8 qubits). Expect
minutes-to-hours per cell, not the seconds the classical arms took. Section 2
measures throughput before you commit. The grid is resumable, so a disconnect
costs one cell.

---
## 1. Environment

Same as the main runner. Run cell, restart if it says so, continue below.

In [ ]:
from google.colab import userdata

GITHUB_USER = "RogerMas99"
REPO_NAME = "qrl-dissection"
BRANCH = "main"
try:
    GH_TOKEN = userdata.get("GH_TOKEN")
    REPO_URL = f"https://{GH_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git" if GH_TOKEN else f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
except Exception:
    REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

import sys, subprocess, pathlib, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    RESULTS = pathlib.Path("/content/drive/MyDrive/tfm_qrl/exp01")   # SAME dir as the classical grid
    CODE    = pathlib.Path("/content/qrl-dissection")
else:
    RESULTS = pathlib.Path.cwd() / "results" / "exp01"; CODE = pathlib.Path.cwd()
RESULTS.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    if CODE.exists():
        subprocess.run(["git", "-C", str(CODE), "pull", "--quiet"], check=False)
    else:
        subprocess.run(["git", "clone", "--quiet", "-b", BRANCH, REPO_URL, str(CODE)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE / "requirements.txt")], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(CODE)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "jax", "jaxlib"], check=False)

rev = subprocess.run(["git", "-C", str(CODE), "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True).stdout.strip()
print("code   :", CODE, "@", rev)
print("results:", RESULTS, "  (shared with the classical grid)")

import importlib, sys as _sys
_need = False
if "autoray.autoray" in _sys.modules:
    import autoray.autoray as _aa; _need = not hasattr(_aa, "NumpyMimic")
if "jax" in _sys.modules: _need = True
if _need:
    print("Incompatible modules already loaded -> restarting."); os.kill(os.getpid(), 9)
else:
    print("Clean environment: no restart needed. Continue below.")

### After restarting (if it restarted), run from here

In [ ]:
import sys, pathlib, time
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    RESULTS = pathlib.Path("/content/drive/MyDrive/tfm_qrl/exp01")
    CODE    = pathlib.Path("/content/qrl-dissection")
else:
    RESULTS = pathlib.Path.cwd() / "results" / "exp01"; CODE = pathlib.Path.cwd()
sys.path.insert(0, str(CODE / "src"))

import qrl_dissection
from qrl_dissection import build_arm_config, capacity_ladder
from qrl_dissection.dqn import RunSpec, run_arm, run_grid, GreedyEvalConfig
from qrl_dissection import analysis
print("ready. git:", __import__("subprocess").run(
    ["git","-C",str(CODE),"rev-parse","--short","HEAD"],capture_output=True,text=True).stdout.strip())

---
## 2. Throughput probe — hybrid is expensive

One short hybrid run before committing hours. This estimates per-step cost so you
can size the grid. It runs below `learning_starts=10000`, so it UNDER-estimates:
real cost rises once updates begin.

In [ ]:
t0 = time.time()
_ = run_arm(RunSpec(arm="hybrid_fig4", seed=99, fix_autoreset=True,
                    total_timesteps=1500, tag="probe"),
            outdir=RESULTS / "_probe_hybrid")
dt = time.time() - t0
sps = 1500 / dt
print(f"{sps:.1f} steps/s  ->  60k steps ~ {60_000/sps/60:.1f} min per hybrid run")
print(f"6-cell hybrid grid (FIX-01 x 3 seeds) ~ {6*60_000/sps/60:.0f} min (optimistic)")
print("\nIf that is too long: drop to 2 seeds, or 40k steps (the classical arms")
print("separated dead-vs-alive well before 40k). Adjust STEPS/SEEDS in section 3.")

---
## 3. The hybrid grid

`hybrid_fig4`, FIX-01 on/off, 3 seeds, 60k steps — identical protocol to the
classical grid so the four arms are directly comparable. Writes into the SAME
results dir, so the analysis in section 4 picks up all four arms at once.

Resumable: finished cells are skipped on re-run.

In [ ]:
ARM   = "hybrid_fig4"
SEEDS = [1, 2, 3]     # drop to [1, 2] if throughput is tight
STEPS = 60_000        # drop to 40_000 if tight; classical arms decided well before then

specs = [RunSpec(arm=ARM, seed=seed, fix_autoreset=fix, total_timesteps=STEPS,
                 dqn_kwargs=dict(batch_size=128, buffer_size=10_000, train_frequency=10))
         for fix in (False, True) for seed in SEEDS]

print(f"{len(specs)} hybrid cells into {RESULTS}")
results = run_grid(specs, RESULTS, eval_cfg=GreedyEvalConfig(every_steps=10_000), progress_bar=True)
print(f"\n{sum('error' not in r for r in results)}/{len(results)} ok")
for r in results:
    if "error" in r: print("  FAILED", r["run_name"], r["error"])

---
## 4. All four arms at 60k

Now the comparison is clean: same steps, same seeds, same everything. `best_ma50`
is the headline; `greedy_best` (epsilon=0 evaluation) is primary where present.

In [ ]:
import pandas as pd
df = analysis.to_dataframe(RESULTS)

# order arms by capacity for readability
order = ["paper_linear", "matched_classical", "hybrid_fig4", "oversized_mlp"]
df["arm"] = pd.Categorical(df["arm"], categories=order, ordered=True)
df = df.sort_values(["arm", "fix01", "seed"])
display(df)

print("\n=== best_ma50 by arm x FIX-01 ===")
print(analysis.arm_comparison(RESULTS, metric="best_ma50"))

print("\n=== the comparison that matters: hybrid vs matched_classical (equal budget ~124 params) ===")
for arm in ["matched_classical", "hybrid_fig4"]:
    sub = df[df.arm == arm]
    if len(sub):
        print(f"  {arm:20} best_ma50 {sub.best_ma50.mean():6.1f} +/- {sub.best_ma50.std():5.1f} | "
              f"greedy_best {sub.get('greedy_best', pd.Series([float('nan')])).mean():6.1f}")

In [ ]:
fig = analysis.plot_arms(RESULTS, arms=order, savepath=str(RESULTS / "exp01_all_arms.png"))
fig

---
## 5. Reading it

**Hybrid learns, matched_classical dead** (the expected outcome): at equal
parameter budget the circuit reaches what an equal-size classical block cannot,
under DQN. This is a positive result for the PQC and stronger than the paper,
which never shows the circuit is *necessary*. Note it is specific to this DQN
regime (tf=10, this budget) — say so.

**Both dead**: then 124 parameters is simply too few for either kind of model to
learn CartPole under DQN at tf=10, and the live/dead boundary is about capacity,
not quantum-vs-classical. Still a clean result — it bounds where the dissection
is even identifiable off-policy.

**Hybrid learns AND FIX-01 matters here**: the hybrid is a second live arm, so
unlike the classical arms its FIX-01 delta is interpretable. Watch the seed
variance before claiming an effect — in oversized_mlp a +31.6 delta vanished into
sd 44-106.

Whatever the outcome: paste the section-4 table into `docs/RESULTS-LOG.md` with
the git rev and `verify_env.py` output, commit, and tag `exp01-v1`.

### Threats to validity, still
- n=3; seed dispersion at the top of the range is a factor of two.
- Single seed hybrid at 103k reached 57; these 60k runs may land lower. Compare
  like-for-like (60k vs 60k), not against the old 103k number.
- FIX-02 (OutputScale) is active in the hybrid arm and untested in effect — if
  the hybrid underperforms, that is a live suspect, separate from FIX-01.